In [1]:
# Estrategia Lay Away

import pandas as pd
import numpy as np

In [5]:
data = pd.read_csv("../../data_total/dados_betfair.csv", sep=";")

In [227]:
datatest = data[['League', 'Season', 'Home', 'Away', 'Goals_H_FT', 'Goals_A_FT', 'Odd_H_Back', 'Odd_A_Back', 'Odd_D_Back', 'Odd_A_Lay']].copy()

# Filtro Odd_H_Back menor que 3
#datatest = datatest[datatest['Odd_H_Back'] <= 1.80]

#datatest.to_csv("../test_estrat/data/datatest.csv", index=False, sep=";")

# Criação de variaveis para análise
datatest['VAR1'] = round(np.sqrt((datatest['Odd_H_Back'] - datatest['Odd_A_Back']) ** 2), 2)
datatest['VAR2'] = round(np.degrees(np.arctan((datatest['Odd_A_Back'] - datatest['Odd_H_Back']) / 2)), 2)
datatest['VAR3'] = round(np.degrees(np.arctan((datatest['Odd_D_Back'] - datatest['Odd_A_Back']) / 2)), 2)

# Custo do Gol
datatest['Gol_Custo'] = round((datatest['Odd_A_Lay'] - 1) / 10, 2)

# Verificadbo se o time da casa venceu ou não
datatest['Lay_Away'] = np.where((datatest['Goals_H_FT'] >= datatest['Goals_A_FT']), 1, 0)

# Definir o valor da aposta
STAKE = 1
COMISSAO = 0.065

# Criar o 'profit'
datatest['Profit'] = round(datatest.apply(lambda row: 
    -STAKE * (row['Odd_A_Lay'] - 1) if row['Lay_Away'] == 0  
    else STAKE * (1 - COMISSAO), 
    axis=1), 2)

# fILTRO
#datatest = datatest[(datatest['VAR1'] >= 6.50) & (datatest['VAR2'] >= 68) & (datatest['VAR3'] >= -74) & (datatest['Odd_A_Lay'] > 8) & (datatest['Odd_A_Lay'] <= 10)]
datatest = datatest[(datatest['Gol_Custo'] >= 0.77) & (datatest['VAR1'] >= 6.50) & (datatest['VAR3'] >= -71) & (datatest['Odd_A_Lay'] <= 20)]

# Quantidade de apostas
num_apostas = len(datatest)
print(f"Número de apostas: {num_apostas}")

# Quantidade de apostas vencedoras
num_vitorias = datatest['Lay_Away'].sum()
print(f"Número de apostas vencedoras: {num_vitorias}")

# Taxa de acerto
taxa_acerto = (num_vitorias / num_apostas) * 100
print(f"Taxa de acerto: {taxa_acerto:.2f}%")

# Somar o lucro total
lucro_total = datatest['Profit'].sum()
print(f"Lucro total: {lucro_total}")

# Calcular Yield
yield_value = (lucro_total / (num_apostas * STAKE)) * 100
#print(f"Yield: {yield_value:.2f}%")

# Medir o Drawdown
datatest['Cumulative_Profit'] = datatest['Profit'].cumsum()
datatest['Drawdown'] = datatest['Cumulative_Profit'] - datatest['Cumulative_Profit'].cummax()
max_drawdown = datatest['Drawdown'].min()
print(f"Drawdown máximo: {round(max_drawdown, 2)}%")

datatest.head()

Número de apostas: 312
Número de apostas vencedoras: 294
Taxa de acerto: 94.23%
Lucro total: 105.05999999999992
Drawdown máximo: -20.66%


,League,Season,Home,Away,Goals_H_FT,Goals_A_FT,Odd_H_Back,Odd_A_Back,Odd_D_Back,Odd_A_Lay,VAR1,VAR2,VAR3,Gol_Custo,Lay_Away,Profit,Cumulative_Profit,Drawdown
20,ITALY 1,2023/2024,Juventus,Genoa,0,0,1.52,8.8,4.3,9.0,7.28,74.64,-66.04,0.80,1,0.94,0.94,0.0
25,FRANCE 1,2023/2024,Monaco,Lorient,2,2,1.36,10.5,5.9,11.0,9.14,77.66,-66.50,1.00,1,0.94,1.88,0.0
41,SPAIN 1,2023/2024,Barcelona,Las Palmas,1,0,1.27,12.5,7.0,13.0,11.23,79.90,-70.02,1.20,1,0.94,2.82,0.0
45,ENGLAND 1,2023/2024,Chelsea,Burnley,2,2,1.32,11.0,6.4,11.5,9.68,78.33,-66.50,1.05,1,0.94,3.76,0.0
47,ENGLAND 1,2023/2024,Tottenham,Luton,2,1,1.24,13.5,8.2,14.0,12.26,80.73,-69.33,1.30,1,0.94,4.70,0.0
